# Construcción del dataset de modelado

Este notebook construye el dataset utilizado posteriormente para el entrenamiento de los modelos de predicción de calidad del aire.

El proceso parte de los datos procesados por la ETL almacenados en DuckDB e integra, para cada estación de calidad del aire:

- concentración diaria del contaminante objetivo;
- información meteorológica;
- información de calendario;
- estacionalidad mensual;
- concentraciones históricas del propio contaminante mediante variables lag.

El resultado final es un dataset diario por estación preparado para su utilización en el notebook de modelado.

En este notebook se utiliza **NO₂** como contaminante objetivo para desarrollar la primera versión del pipeline.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURACIÓN GENERAL DEL MODELO
# ============================================================

# Contaminantes objetivo del TFM
MAGNITUDES = {
    "NO2": 8,
    "PM2.5": 9,
    "PM10": 10,
    "O3": 14,
}

# Contaminante con el que desarrollamos la plantilla
CONTAMINANTE = "NO2"
MAGNITUD = MAGNITUDES[CONTAMINANTE]

# Lags iniciales.
# Más adelante probaremos distintas configuraciones mediante
# validación temporal.
LAGS = [1, 2, 3, 7]

# 2024 se reserva como test final
TEST_YEAR = 2024

# Semilla para reproducibilidad
RANDOM_STATE = 42


print(f"Contaminante seleccionado: {CONTAMINANTE}")
print(f"Magnitud: {MAGNITUD}")

Contaminante seleccionado: NO2
Magnitud: 8


In [2]:
# ============================================================
# LOCALIZACIÓN DEL PROYECTO
# ============================================================

ROOT = Path.cwd()

while not (ROOT / "src" / "database" / "TFM.duckdb").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError(
            "No se ha encontrado src/database/TFM.duckdb"
        )
    ROOT = ROOT.parent

DB_PATH = ROOT / "src" / "database" / "TFM.duckdb"

print("Raíz del proyecto:", ROOT)
print("Base de datos:", DB_PATH)

Raíz del proyecto: c:\Users\borja\Desktop\Master\TFM
Base de datos: c:\Users\borja\Desktop\Master\TFM\src\database\TFM.duckdb


In [3]:
# ============================================================
# CONEXIÓN A DUCKDB
# ============================================================

# Solo lectura porque desde modelado no queremos modificar
# los datos generados por la ETL.
con = duckdb.connect(
    str(DB_PATH),
    read_only=True
)

print("Conexión a DuckDB realizada correctamente.")

Conexión a DuckDB realizada correctamente.


In [4]:
# ============================================================
# COMPROBACIÓN DE FUENTES
# ============================================================

tables = con.execute(
    "SHOW ALL TABLES"
).fetchdf()

display(
    tables[
        ["schema", "name"]
    ].sort_values(
        ["schema", "name"]
    )
)

,schema,name
0,enriched,calidad_aire_final
1,enriched,dim_estaciones_aire
2,enriched,dim_estaciones_meteo
3,enriched,meteo_final
4,normalized,calidad_aire_historico
5,normalized,dim_variable_aire
6,normalized,meteo_aemet_historico
7,normalized,seed_festivos
8,normalized,seed_rango_variables_aire
9,normalized,seed_variable_aire


## Carga y validación de los datos de calidad del aire

Se extraen de la base de datos las observaciones correspondientes al contaminante objetivo.

Antes de construir nuevas variables se comprueba la estructura del dataset, el número de estaciones disponibles, el periodo temporal cubierto, la existencia de valores nulos o duplicados y la continuidad temporal de las observaciones.

Estas comprobaciones permiten detectar posibles problemas en los datos antes de realizar la integración con las variables meteorológicas.

In [5]:
# ============================================================
# CARGA DEL CONTAMINANTE OBJETIVO
# ============================================================

aire = con.execute(
    """
    SELECT
        fecha,
        estacion,
        uom_value AS contaminante_t,
        es_laborable_madrid_ciudad
    FROM enriched.calidad_aire_final
    WHERE magnitud = ?
    ORDER BY estacion, fecha
    """,
    [MAGNITUD]
).fetchdf()

display(aire.head(10))

,fecha,estacion,contaminante_t,es_laborable_madrid_ciudad
0,2020-01-01,8,65.666667,False
1,2020-01-02,8,68.083333,True
2,2020-01-03,8,73.458333,True
3,2020-01-04,8,42.347826,False
4,2020-01-05,8,51.291667,False
5,2020-01-06,8,60.083333,False
6,2020-01-07,8,85.130435,True
7,2020-01-08,8,99.791667,True
8,2020-01-09,8,77.958333,True
9,2020-01-10,8,54.500000,True


In [6]:
# ============================================================
# COMPROBACIÓN GENERAL DEL DATASET
# ============================================================

print(f"Contaminante: {CONTAMINANTE}")
print(f"Magnitud: {MAGNITUD}")

print(f"\nNúmero de observaciones: {len(aire):,}")
print(f"Número de estaciones: {aire['estacion'].nunique()}")

print(
    f"Periodo: {aire['fecha'].min()} "
    f"→ {aire['fecha'].max()}"
)

print("\nObservaciones por estación:")

display(
    aire.groupby("estacion")
        .size()
        .rename("n_observaciones")
        .to_frame()
)

Contaminante: NO2
Magnitud: 8

Número de observaciones: 40,194
Número de estaciones: 22
Periodo: 2020-01-01 00:00:00 → 2024-12-31 00:00:00

Observaciones por estación:


,n_observaciones
estacion,
8,1827
16,1827
17,1827
18,1827
24,1827
27,1827
35,1827
36,1827
38,1827


In [7]:
# ============================================================
# COMPROBACIÓN DE DUPLICADOS
# ============================================================

duplicados = aire.duplicated(
    subset=["estacion", "fecha"]
).sum()

print(
    f"Duplicados estación-fecha: {duplicados}"
)

Duplicados estación-fecha: 0


In [8]:
# ============================================================
# COMPROBACIÓN DE NULOS
# ============================================================

nulos = (
    aire
    .isna()
    .sum()
    .to_frame("n_nulos")
)

nulos["porcentaje"] = (
    nulos["n_nulos"] / len(aire) * 100
)

display(nulos)

,n_nulos,porcentaje
fecha,0,0.0
estacion,0,0.0
contaminante_t,0,0.0
es_laborable_madrid_ciudad,0,0.0


In [9]:
# ============================================================
# COMPROBACIÓN DE CONTINUIDAD TEMPORAL
# ============================================================

aire["fecha"] = pd.to_datetime(aire["fecha"])

comprobacion_fechas = (
    aire.sort_values(["estacion", "fecha"])
        .assign(
            diferencia_dias=lambda x:
                x.groupby("estacion")["fecha"]
                 .diff()
                 .dt.days
        )
)

saltos = comprobacion_fechas[
    comprobacion_fechas["diferencia_dias"] > 1
]

print(f"Número de saltos temporales: {len(saltos)}")

display(saltos.head(20))

Número de saltos temporales: 0


,fecha,estacion,contaminante_t,es_laborable_madrid_ciudad,diferencia_dias


In [10]:
# ============================================================
# RESUMEN
# ============================================================

print("=" * 50)
print("RESUMEN DEL DATASET DE AIRE")
print("=" * 50)

print(f"Contaminante:       {CONTAMINANTE}")
print(f"Magnitud:           {MAGNITUD}")
print(f"Observaciones:      {len(aire):,}")
print(f"Estaciones:         {aire['estacion'].nunique()}")
print(f"Fecha inicial:      {aire['fecha'].min().date()}")
print(f"Fecha final:        {aire['fecha'].max().date()}")
print(f"Duplicados:         {duplicados}")
print(f"Saltos temporales:  {len(saltos)}")
print(f"Nulos contaminante: {aire['contaminante_t'].isna().sum()}")

RESUMEN DEL DATASET DE AIRE
Contaminante:       NO2
Magnitud:           8
Observaciones:      40,194
Estaciones:         22
Fecha inicial:      2020-01-01
Fecha final:        2024-12-31
Duplicados:         0
Saltos temporales:  0
Nulos contaminante: 0


## Asignación de información meteorológica

Las estaciones de calidad del aire y las estaciones meteorológicas no se encuentran necesariamente en la misma localización.

Para incorporar las condiciones meteorológicas a cada estación de calidad del aire se utiliza la información de distancia entre estaciones disponible en la base de datos.

Inicialmente se analiza la estación meteorológica más cercana y la disponibilidad de las distintas variables. Dado que no todas las estaciones meteorológicas registran todas las variables, la asignación definitiva se realiza **por variable meteorológica**.

De esta forma, para cada estación de calidad del aire y cada variable meteorológica se selecciona la estación meteorológica más cercana que disponga de dicha variable.

In [11]:
# ============================================================
# ASIGNACIÓN DE ESTACIÓN METEOROLÓGICA MÁS CERCANA
# ============================================================

mapping_meteo = con.execute(
    """
    WITH meteo_disponibles AS (
        SELECT DISTINCT estacion_id
        FROM enriched.meteo_final
    ),

    candidatos AS (
        SELECT
            a.ESTACION AS estacion,
            kv.key AS estacion_meteo,
            kv.value AS distancia_meteo_km
        FROM enriched.dim_estaciones_aire a,
             UNNEST(map_entries(a.distancias_meteo)) AS t(kv)

        INNER JOIN meteo_disponibles m
            ON kv.key = m.estacion_id
    ),

    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY estacion
                ORDER BY distancia_meteo_km
            ) AS rn
        FROM candidatos
    )

    SELECT
        estacion,
        estacion_meteo,
        distancia_meteo_km
    FROM ranked
    WHERE rn = 1
    ORDER BY estacion
    """
).fetchdf()

display(mapping_meteo)

,estacion,estacion_meteo,distancia_meteo_km
0,8,3195,1.20
1,16,3195,4.56
2,17,3200,5.48
3,18,3195,4.95
4,24,3196,5.85
5,27,3129,1.87
6,35,3195,2.33
7,36,3195,2.78
8,38,3195,4.48
9,39,3195,7.97


In [12]:
estaciones_no2 = set(aire["estacion"].unique())
estaciones_mapping = set(mapping_meteo["estacion"].unique())

assert estaciones_no2.issubset(estaciones_mapping)

print("Todas las estaciones de NO2 tienen estación meteo asignada.")


Todas las estaciones de NO2 tienen estación meteo asignada.


In [13]:
# ============================================================
# CARGA DE METEOROLOGÍA
# ============================================================

meteo_long = con.execute(
    """
    SELECT
        fecha,
        estacion_id,
        variable,
        uom_value
    FROM enriched.meteo_final
    ORDER BY estacion_id, fecha, variable
    """
).fetchdf()

meteo_long["fecha"] = pd.to_datetime(meteo_long["fecha"])

display(meteo_long.head())

,fecha,estacion_id,variable,uom_value
0,2020-01-01,3129,direccion_racha_max,220.0
1,2020-01-01,3129,humedad_max,95.0
2,2020-01-01,3129,humedad_media,79.0
3,2020-01-01,3129,humedad_min,53.0
4,2020-01-01,3129,insolacion,8.3


In [14]:
# ============================================================
# PIVOT DE METEOROLOGÍA
# ============================================================

meteo = (
    meteo_long
    .pivot(
        index=["fecha", "estacion_id"],
        columns="variable",
        values="uom_value"
    )
    .reset_index()
)

meteo.columns.name = None

display(meteo.head())

,fecha,estacion_id,direccion_racha_max,humedad_max,humedad_media,humedad_min,insolacion,precipitacion,presion_max,presion_min,temp_max,temp_media,temp_min,viento_racha,viento_velocidad
0,2020-01-01,3129,220.0,95.0,79.0,53.0,8.3,0.0,964.6,961.5,12.8,5.4,-2.1,3.6,0.8
1,2020-01-01,3195,50.0,80.0,65.0,48.0,NaN,0.0,954.2,952.0,11.2,6.6,1.9,3.3,0.3
2,2020-01-01,3196,360.0,100.0,78.0,49.0,8.7,0.0,952.2,949.5,13.5,7.2,0.8,3.1,1.4
3,2020-01-01,3200,250.0,NaN,81.0,NaN,7.8,0.0,960.5,957.6,12.8,6.1,-0.6,2.5,0.0
4,2020-01-02,3129,200.0,95.0,78.0,50.0,8.2,0.0,963.7,960.9,12.9,5.3,-2.3,5.0,1.1


In [15]:
# Comprobar disponibilidad de cada variable por estación meteorológica

disponibilidad_meteo = (
    meteo_long
    .groupby(["estacion_id", "variable"])
    .size()
    .unstack(fill_value=0)
)

display(disponibilidad_meteo)

variable,direccion_racha_max,humedad_max,humedad_media,humedad_min,insolacion,precipitacion,presion_max,presion_min,temp_max,temp_media,temp_min,viento_racha,viento_velocidad
estacion_id,,,,,,,,,,,,,
3129,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827
3195,1827,1827,1827,1827,0,1827,1827,1827,1827,1827,1827,1827,1827
3196,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827,1827
3200,1827,0,1827,0,1827,1827,1827,1827,1827,1827,1827,1827,1827


In [16]:
for estacion in sorted(meteo["estacion_id"].unique()):
    print(f"\nEstación {estacion}")
    
    faltantes = (
        meteo.loc[
            meteo["estacion_id"] == estacion
        ]
        .isna()
        .sum()
    )
    
    print(faltantes[faltantes > 0])


Estación 3129
Series([], dtype: int64)

Estación 3195
insolacion    1827
dtype: int64

Estación 3196
Series([], dtype: int64)

Estación 3200
humedad_max    1827
humedad_min    1827
dtype: int64


## Construcción del dataset unificado

Una vez asignada la información meteorológica correspondiente a cada estación de calidad del aire, se integran ambos conjuntos de datos mediante las variables `estacion` y `fecha`.

La unión se valida como una relación uno a uno para garantizar que exista como máximo una observación por estación y día.

Posteriormente se comprueba nuevamente la dimensión del dataset, el periodo temporal, los duplicados y la presencia de valores nulos antes de realizar la ingeniería de variables.

In [17]:
# ============================================================
# MAPPING AIRE → ESTACIÓN METEO MÁS CERCANA POR VARIABLE
# ============================================================

mapping_meteo_variable = con.execute(
    """
    WITH disponibilidad AS (
        -- Variables realmente disponibles en cada estación meteo
        SELECT DISTINCT
            estacion_id,
            variable
        FROM enriched.meteo_final
    ),

    candidatos AS (
        SELECT
            a.ESTACION AS estacion,
            d.variable,
            kv.key AS estacion_meteo,
            kv.value AS distancia_meteo_km

        FROM enriched.dim_estaciones_aire a,
             UNNEST(map_entries(a.distancias_meteo)) AS t(kv)

        INNER JOIN disponibilidad d
            ON kv.key = d.estacion_id
    ),

    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY estacion, variable
                ORDER BY distancia_meteo_km
            ) AS rn
        FROM candidatos
    )

    SELECT
        estacion,
        variable,
        estacion_meteo,
        distancia_meteo_km
    FROM ranked
    WHERE rn = 1
    ORDER BY estacion, variable
    """
).fetchdf()

# Nos quedamos solo con las estaciones utilizadas por NO2
mapping_meteo_variable = mapping_meteo_variable[
    mapping_meteo_variable["estacion"].isin(
        aire["estacion"].unique()
    )
].copy()

display(mapping_meteo_variable.head(30))

,estacion,variable,estacion_meteo,distancia_meteo_km
0,8,direccion_racha_max,3195,1.20
1,8,humedad_max,3195,1.20
2,8,humedad_media,3195,1.20
3,8,humedad_min,3195,1.20
4,8,insolacion,3196,10.18
5,8,precipitacion,3195,1.20
6,8,presion_max,3195,1.20
7,8,presion_min,3195,1.20
8,8,temp_max,3195,1.20
9,8,temp_media,3195,1.20


In [18]:
# ============================================================
# APLICAR MAPPING METEO POR VARIABLE
# ============================================================

meteo_asignado = mapping_meteo_variable.merge(
    meteo_long,
    left_on=["estacion_meteo", "variable"],
    right_on=["estacion_id", "variable"],
    how="left"
)

display(meteo_asignado.head())

,estacion,variable,estacion_meteo,distancia_meteo_km,fecha,estacion_id,uom_value
0,8,direccion_racha_max,3195,1.2,2020-01-01,3195,50.0
1,8,direccion_racha_max,3195,1.2,2020-01-02,3195,230.0
2,8,direccion_racha_max,3195,1.2,2020-01-03,3195,95.0
3,8,direccion_racha_max,3195,1.2,2020-01-04,3195,40.0
4,8,direccion_racha_max,3195,1.2,2020-01-05,3195,50.0


In [19]:
# ============================================================
# PIVOT METEO FINAL PARA CADA ESTACIÓN DE AIRE
# ============================================================

meteo_modelo = (
    meteo_asignado
    .pivot(
        index=["estacion", "fecha"],
        columns="variable",
        values="uom_value"
    )
    .reset_index()
)

meteo_modelo.columns.name = None

display(meteo_modelo.head())

,estacion,fecha,direccion_racha_max,humedad_max,humedad_media,humedad_min,insolacion,precipitacion,presion_max,presion_min,temp_max,temp_media,temp_min,viento_racha,viento_velocidad
0,8,2020-01-01,50.0,80.0,65.0,48.0,8.7,0.0,954.2,952.0,11.2,6.6,1.9,3.3,0.3
1,8,2020-01-02,230.0,89.0,77.0,63.0,8.3,0.0,953.7,951.0,10.5,6.0,1.6,3.3,0.6
2,8,2020-01-03,95.0,89.0,83.0,74.0,1.4,0.0,954.8,952.2,6.0,3.8,1.5,5.0,0.6
3,8,2020-01-04,40.0,77.0,58.0,44.0,8.9,0.0,955.9,952.5,11.2,7.0,2.8,9.4,2.5
4,8,2020-01-05,50.0,74.0,53.0,42.0,8.6,0.0,953.3,949.2,10.9,6.2,1.5,5.3,0.8


In [20]:
# ============================================================
# COMPROBACIÓN FINAL DE NULOS
# ============================================================

print("Shape:", meteo_modelo.shape)

print("\nNulos por variable:")

display(
    meteo_modelo
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("n_nulos")
)

print(
    "\nTOTAL DE NULOS:",
    meteo_modelo.isna().sum().sum()
)

Shape: (40194, 15)

Nulos por variable:


,n_nulos
estacion,0
fecha,0
direccion_racha_max,0
humedad_max,0
humedad_media,0
humedad_min,0
insolacion,0
precipitacion,0
presion_max,0
presion_min,0



TOTAL DE NULOS: 0


In [21]:
# ============================================================
# UNIÓN AIRE + METEOROLOGÍA
# ============================================================

df_modelo = aire.merge(
    meteo_modelo,
    on=["estacion", "fecha"],
    how="left",
    validate="one_to_one"
)

df_modelo = df_modelo.sort_values(
    ["estacion", "fecha"]
).reset_index(drop=True)

print("Shape aire:", aire.shape)
print("Shape meteo:", meteo_modelo.shape)
print("Shape modelo:", df_modelo.shape)

display(df_modelo.head())

Shape aire: (40194, 4)
Shape meteo: (40194, 15)
Shape modelo: (40194, 17)


,fecha,estacion,contaminante_t,es_laborable_madrid_ciudad,direccion_racha_max,humedad_max,humedad_media,humedad_min,insolacion,precipitacion,presion_max,presion_min,temp_max,temp_media,temp_min,viento_racha,viento_velocidad
0,2020-01-01,8,65.666667,False,50.0,80.0,65.0,48.0,8.7,0.0,954.2,952.0,11.2,6.6,1.9,3.3,0.3
1,2020-01-02,8,68.083333,True,230.0,89.0,77.0,63.0,8.3,0.0,953.7,951.0,10.5,6.0,1.6,3.3,0.6
2,2020-01-03,8,73.458333,True,95.0,89.0,83.0,74.0,1.4,0.0,954.8,952.2,6.0,3.8,1.5,5.0,0.6
3,2020-01-04,8,42.347826,False,40.0,77.0,58.0,44.0,8.9,0.0,955.9,952.5,11.2,7.0,2.8,9.4,2.5
4,2020-01-05,8,51.291667,False,50.0,74.0,53.0,42.0,8.6,0.0,953.3,949.2,10.9,6.2,1.5,5.3,0.8


In [22]:
# ============================================================
# VALIDACIÓN DEL DATASET UNIFICADO
# ============================================================

print("Filas:", len(df_modelo))
print("Estaciones:", df_modelo["estacion"].nunique())

print(
    "Periodo:",
    df_modelo["fecha"].min(),
    "→",
    df_modelo["fecha"].max()
)

print(
    "Duplicados estación-fecha:",
    df_modelo.duplicated(["estacion", "fecha"]).sum()
)

print("\nNulos:")

display(
    df_modelo
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("n_nulos")
)

Filas: 40194
Estaciones: 22
Periodo: 2020-01-01 00:00:00 → 2024-12-31 00:00:00
Duplicados estación-fecha: 0

Nulos:


,n_nulos
fecha,0
estacion,0
contaminante_t,0
es_laborable_madrid_ciudad,0
direccion_racha_max,0
humedad_max,0
humedad_media,0
humedad_min,0
insolacion,0
precipitacion,0


## Incorporación de estacionalidad mensual

Para representar la componente estacional se incorpora el mes del año mediante una codificación cíclica basada en seno y coseno.

Representar directamente el mes como un número entero introduciría una distancia artificial entre diciembre (`12`) y enero (`1`), a pesar de ser meses consecutivos.

Por ello se utilizan las transformaciones:

$$
mes_{sin} = \sin\left(\frac{2\pi \cdot mes}{12}\right)
$$

$$
mes_{cos} = \cos\left(\frac{2\pi \cdot mes}{12}\right)
$$

Esta representación conserva la naturaleza circular del calendario anual y permite al modelo capturar patrones estacionales.

In [23]:
# ============================================================
# ESTACIONALIDAD MENSUAL
# ============================================================

import numpy as np

df_modelo["mes"] = df_modelo["fecha"].dt.month

df_modelo["mes_sin"] = np.sin(
    2 * np.pi * df_modelo["mes"] / 12
)

df_modelo["mes_cos"] = np.cos(
    2 * np.pi * df_modelo["mes"] / 12
)

display(
    df_modelo[
        ["fecha", "mes", "mes_sin", "mes_cos"]
    ].head(15)
)

,fecha,mes,mes_sin,mes_cos
0,2020-01-01,1,0.5,0.866025
1,2020-01-02,1,0.5,0.866025
2,2020-01-03,1,0.5,0.866025
3,2020-01-04,1,0.5,0.866025
4,2020-01-05,1,0.5,0.866025
5,2020-01-06,1,0.5,0.866025
6,2020-01-07,1,0.5,0.866025
7,2020-01-08,1,0.5,0.866025
8,2020-01-09,1,0.5,0.866025
9,2020-01-10,1,0.5,0.866025


## Construcción de variables temporales

Las concentraciones de contaminantes atmosféricos presentan dependencia temporal, por lo que las observaciones recientes pueden aportar información relevante para predecir la concentración del día siguiente.

Para representar esta dependencia se generan, de forma independiente para cada estación, las siguientes variables:

- `lag_1`: concentración observada el día anterior.
- `lag_2`: concentración observada dos días antes.
- `lag_3`: concentración observada tres días antes.
- `lag_7`: concentración observada una semana antes.

Los lags se calculan agrupando por estación y manteniendo el orden cronológico, evitando mezclar información entre estaciones diferentes.

Estas variables permiten utilizar únicamente información histórica del contaminante para predecir la concentración del día objetivo.

In [24]:
# ============================================================
# LAGS TEMPORALES DE NO2
# ============================================================

# Nos aseguramos de que los datos estén ordenados temporalmente
df_modelo = (
    df_modelo
    .sort_values(["estacion", "fecha"])
    .reset_index(drop=True)
)

# Lags de NO2 por estación
for lag in [1, 2, 3, 7]:
    df_modelo[f"no2_lag_{lag}"] = (
        df_modelo
        .groupby("estacion")["contaminante_t"]
        .shift(lag)
    )

display(
    df_modelo[
        [
            "fecha",
            "estacion",
            "contaminante_t",
            "no2_lag_1",
            "no2_lag_2",
            "no2_lag_3",
            "no2_lag_7"
        ]
    ].head(15)
)

,fecha,estacion,contaminante_t,no2_lag_1,no2_lag_2,no2_lag_3,no2_lag_7
0,2020-01-01,8,65.666667,NaN,NaN,NaN,NaN
1,2020-01-02,8,68.083333,65.666667,NaN,NaN,NaN
2,2020-01-03,8,73.458333,68.083333,65.666667,NaN,NaN
3,2020-01-04,8,42.347826,73.458333,68.083333,65.666667,NaN
4,2020-01-05,8,51.291667,42.347826,73.458333,68.083333,NaN
5,2020-01-06,8,60.083333,51.291667,42.347826,73.458333,NaN
6,2020-01-07,8,85.130435,60.083333,51.291667,42.347826,NaN
7,2020-01-08,8,99.791667,85.130435,60.083333,51.291667,65.666667
8,2020-01-09,8,77.958333,99.791667,85.130435,60.083333,68.083333
9,2020-01-10,8,54.500000,77.958333,99.791667,85.130435,73.458333


In [25]:
# ============================================================
# COMPROBACIÓN DE LAGS
# ============================================================

lags = [
    "no2_lag_1",
    "no2_lag_2",
    "no2_lag_3",
    "no2_lag_7"
]

display(
    df_modelo[lags]
    .isna()
    .sum()
    .to_frame("n_nulos")
)

,n_nulos
no2_lag_1,22
no2_lag_2,44
no2_lag_3,66
no2_lag_7,154


In [26]:
# ============================================================
# COMPROBAR LAGS EN CAMBIO DE AÑO
# ============================================================

comprobacion = df_modelo[
    (df_modelo["estacion"] == 8) &
    (df_modelo["fecha"].between("2020-12-27", "2021-01-05"))
][
    [
        "fecha",
        "contaminante_t",
        "no2_lag_1",
        "no2_lag_2",
        "no2_lag_3",
        "no2_lag_7"
    ]
]

display(comprobacion)

,fecha,contaminante_t,no2_lag_1,no2_lag_2,no2_lag_3,no2_lag_7
361,2020-12-27,39.750000,49.916667,21.791667,31.347826,39.166667
362,2020-12-28,32.750000,39.750000,49.916667,21.791667,49.869565
363,2020-12-29,36.913043,32.750000,39.750000,49.916667,53.750000
364,2020-12-30,41.181818,36.913043,32.750000,39.750000,46.875000
365,2020-12-31,35.041667,41.181818,36.913043,32.750000,31.347826
366,2021-01-01,22.916667,35.041667,41.181818,36.913043,21.791667
367,2021-01-02,38.625000,22.916667,35.041667,41.181818,49.916667
368,2021-01-03,39.041667,38.625000,22.916667,35.041667,39.750000
369,2021-01-04,47.956522,39.041667,38.625000,22.916667,32.750000
370,2021-01-05,63.083333,47.956522,39.041667,38.625000,36.913043


### Observaciones sin histórico suficiente

La creación de variables lag genera necesariamente valores nulos al comienzo de la serie temporal de cada estación, ya que no existen observaciones anteriores suficientes para construir todos los retardos.

Dado que el mayor retardo utilizado es de siete días, las primeras observaciones de cada estación no pueden utilizarse para el modelado.

Estas filas se eliminan únicamente una vez construidas las variables temporales, conservando el resto de observaciones disponibles.

In [27]:
# ============================================================
# ELIMINAR OBSERVACIONES SIN HISTÓRICO SUFICIENTE
# ============================================================

lags = [
    "no2_lag_1",
    "no2_lag_2",
    "no2_lag_3",
    "no2_lag_7"
]

df_modelo_final = (
    df_modelo
    .dropna(subset=lags)
    .reset_index(drop=True)
)

print("Filas originales:", len(df_modelo))
print("Filas finales:", len(df_modelo_final))
print("Filas eliminadas:", len(df_modelo) - len(df_modelo_final))

print(
    "Nulos en lags:",
    df_modelo_final[lags].isna().sum().sum()
)

print(
    "Periodo final:",
    df_modelo_final["fecha"].min(),
    "→",
    df_modelo_final["fecha"].max()
)

Filas originales: 40194
Filas finales: 40040
Filas eliminadas: 154
Nulos en lags: 0
Periodo final: 2020-01-08 00:00:00 → 2024-12-31 00:00:00


In [28]:
# ============================================================
# VISTA DEL DATASET FINAL
# ============================================================

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(df_modelo_final.head(30))

,fecha,estacion,contaminante_t,es_laborable_madrid_ciudad,direccion_racha_max,humedad_max,humedad_media,humedad_min,insolacion,precipitacion,presion_max,presion_min,temp_max,temp_media,temp_min,viento_racha,viento_velocidad,mes,mes_sin,mes_cos,no2_lag_1,no2_lag_2,no2_lag_3,no2_lag_7
0,2020-01-08,8,99.791667,True,90.0,72.0,47.0,27.0,8.8,0.0,951.8,949.4,13.5,7.6,1.8,3.3,0.3,1,0.500000,0.866025,85.130435,60.083333,51.291667,65.666667
1,2020-01-09,8,77.958333,True,300.0,75.0,59.0,49.0,8.6,0.0,950.9,946.3,11.5,7.8,4.0,9.4,1.1,1,0.500000,0.866025,99.791667,85.130435,60.083333,68.083333
2,2020-01-10,8,54.500000,True,40.0,73.0,50.0,37.0,8.9,0.0,952.0,947.0,10.8,7.5,4.2,9.7,3.3,1,0.500000,0.866025,77.958333,99.791667,85.130435,73.458333
3,2020-01-11,8,56.625000,False,50.0,69.0,52.0,38.0,9.2,0.0,954.0,950.8,10.4,6.2,2.0,6.1,1.4,1,0.500000,0.866025,54.500000,77.958333,99.791667,42.347826
4,2020-01-12,8,70.375000,False,170.0,76.0,69.0,56.0,9.1,0.0,953.6,949.8,9.0,4.2,-0.7,3.1,1.1,1,0.500000,0.866025,56.625000,54.500000,77.958333,51.291667
5,2020-01-13,8,79.000000,True,240.0,77.0,65.0,52.0,5.7,0.0,950.6,945.5,8.5,4.0,-0.5,5.6,1.1,1,0.500000,0.866025,70.375000,56.625000,54.500000,60.083333
6,2020-01-14,8,71.125000,True,220.0,85.0,68.0,57.0,7.4,0.0,946.2,943.7,8.4,4.0,-0.5,6.4,0.8,1,0.500000,0.866025,79.000000,70.375000,56.625000,85.130435
7,2020-01-15,8,78.916667,True,70.0,84.0,75.0,63.0,0.0,0.0,949.6,943.8,8.3,5.8,3.3,4.7,0.6,1,0.500000,0.866025,71.125000,79.000000,70.375000,99.791667
8,2020-01-16,8,65.416667,True,260.0,75.0,69.0,49.0,2.5,0.0,949.2,947.0,12.8,10.0,7.2,8.3,1.1,1,0.500000,0.866025,78.916667,71.125000,79.000000,77.958333
9,2020-01-17,8,52.375000,True,260.0,94.0,79.0,55.0,6.7,0.8,953.2,948.6,12.5,8.6,4.8,9.2,1.7,1,0.500000,0.866025,65.416667,78.916667,71.125000,54.500000


## Dataset final de modelado

Tras integrar las distintas fuentes y construir las variables temporales y estacionales, se obtiene el dataset definitivo que será utilizado en la fase de modelado.

Antes de finalizar se comprueba que:

- no existen duplicados por estación y fecha;
- no quedan valores nulos;
- todas las estaciones mantienen observaciones durante el periodo disponible;
- las variables temporales disponen del histórico necesario.

El dataset se almacena en formato Parquet para ser cargado directamente desde el notebook de modelado.

In [29]:
# ============================================================
# GUARDAR DATASET FINAL DE MODELADO
# ============================================================

OUTPUT_PATH = ROOT / "data" / "modeling" / "dataset_no2.parquet"

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_modelo_final.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Dataset guardado correctamente:")
print(OUTPUT_PATH)

print("\nShape:", df_modelo_final.shape)

Dataset guardado correctamente:
c:\Users\borja\Desktop\Master\TFM\data\modeling\dataset_no2.parquet

Shape: (40040, 24)


In [30]:
# ============================================================
# VALIDACIÓN FINAL DEL DATASET GUARDADO
# ============================================================

df_check = pd.read_parquet(OUTPUT_PATH)

print("Shape:", df_check.shape)
print("Duplicados estación-fecha:",
      df_check.duplicated(["estacion", "fecha"]).sum())
print("Nulos totales:", df_check.isna().sum().sum())

print(
    "Periodo:",
    df_check["fecha"].min(),
    "→",
    df_check["fecha"].max()
)

print("Estaciones:", df_check["estacion"].nunique())

Shape: (40040, 24)
Duplicados estación-fecha: 0
Nulos totales: 0
Periodo: 2020-01-08 00:00:00 → 2024-12-31 00:00:00
Estaciones: 22
